# R3-CAP · Capstone — proyecto integrador de IA aplicada — Lección

**Bootcamp de Datos para Funcionarios Públicos · Formación Pública**
Línea C · *IA Aplicada* · Semana 6–7 · Se ejecuta en Google Colab.

> 🚀 **Cómo se trabaja:** lee, ejecuta cada celda con **▶︎** (o `Shift`+`Enter`) y completa los `TODO`. Cada ejercicio termina en una **celda de chequeo** que muestra ✅ si está bien o una pista si todavía no.

---

## ¿Qué vas a lograr?

1. Construir un mini **RAG**: recuperar + responder con contexto.
2. Trabajar con un **LLM en modo caché** (verificable sin red).
3. Documentar el **uso responsable** (costo, privacidad, groundedness).
4. Entregar un proyecto **reproducible**.

**Competencia de salida:** integrar recuperación y generación para responder preguntas públicas con citas y criterio.

**Dato real:** normativa de compras públicas (corpus de ejemplo); reemplazable por el de tu organismo.

**Entregable:** que las **3 celdas de chequeo** muestren ✅.

In [ ]:
# Corpus de ejemplo (normativa de compras públicas). En tu proyecto, reemplázalo por
# los documentos de tu organismo. Sin red y sin LLM real: todo es verificable offline.
CORPUS = [
    "Las licitaciones publicas se rigen por la ley de compras y se publican en MercadoPublico.",
    "El convenio marco permite comprar a proveedores preacordados sin hacer una licitacion.",
    "El trato directo es excepcional y requiere una justificacion formal documentada.",
    "Los proveedores del Estado se clasifican por tamano: micro, pequena, mediana y grande.",
]

def get_llm():
    """Modo CACHE (determinista): NO llama a Gemini ni a la red. Para usar el LLM en vivo,
    define GEMINI_API_KEY y reemplaza esta función por la real (ver R3-00 / R3-02)."""
    def generar(prompt):
        return "Respuesta (modo cache, sin LLM real) basada en el contexto: " + prompt.strip()[:160]
    return generar

print("Corpus:", len(CORPUS), "documentos | get_llm en modo cache")

### El corpus y `get_llm()` (modo caché) están listos arriba. Armemos el RAG.

## 1. Recuperación (la R de RAG)

Un RAG primero **recupera** el documento más relevante. Versión simple: el de mayor solapamiento de palabras con la pregunta.

### ✍️ Ejercicio 1 — Escribe `recuperar`

In [ ]:
def recuperar(pregunta, corpus=CORPUS):
    palabras = set(pregunta.lower().split())
    # TODO: devuelve el documento del corpus con mayor solapamiento de palabras
    return ...

doc = recuperar("¿que es el convenio marco?")
print(doc)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert recuperar("¿que es el convenio marco?") in CORPUS
    assert "convenio" in recuperar("¿que es el convenio marco?").lower()
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'max(corpus, key=lambda d: len(palabras & set(d.lower().split()))).')
    print("   Detalle:", e)

## 2. El LLM en modo caché (determinista)

Para que el notebook sea **verificable sin red**, `get_llm()` devuelve respuestas deterministas. Mismo prompt → misma salida.

### ✍️ Ejercicio 2 — Comprueba que el LLM-caché es determinista

In [ ]:
llm = get_llm()
# TODO: llama dos veces con el mismo prompt y compara
a = ...
b = ...
es_determinista = (a == b)
print("determinista:", es_determinista, "| tipo:", type(a).__name__)

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert es_determinista
    assert isinstance(a, str) and len(a) > 0
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "llm('mismo prompt') dos veces; deben ser iguales.")
    print("   Detalle:", e)

## 3. Responder con contexto (RAG completo)

Une las piezas: **recuperar** el documento y pasárselo al **LLM** como contexto para responder.

### ✍️ Ejercicio 3 — Escribe `responder`

In [ ]:
def responder(pregunta, llm):
    doc = recuperar(pregunta)
    # TODO: arma un prompt con el doc como contexto y la pregunta, y pásalo al llm
    prompt = ...
    return llm(prompt)

respuesta = responder("¿cuando se usa el trato directo?", get_llm())
print(respuesta)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    resp = responder("¿cuando se usa el trato directo?", get_llm())
    assert isinstance(resp, str) and len(resp) > 0
    assert "trato directo" in recuperar("¿cuando se usa el trato directo?").lower()
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'prompt = f"Contexto: \'{doc}\'. Pregunta: {pregunta}" y return llm(prompt).')
    print("   Detalle:", e)

## 4. Ficha de uso responsable

Toda solución de IA en el Estado debe acompañarse de esta ficha:

> **Uso previsto:** _(qué pregunta responde, para quién)_
> **Costo:** usar proveedor con **capa gratuita** (Gemini) + patrón en vivo o caché; estimar tokens por consulta.
> **Privacidad:** los datos de compras son **públicos**; NO enviar datos personales/sensibles a un LLM externo.
> **Verificabilidad (groundedness):** la respuesta **cita** el documento recuperado; sin fuente, no se publica.
> **Supervisión humana:** un funcionario revisa antes de cualquier decisión; el LLM **no decide**.

## Rúbrica de evaluación

| Criterio | Qué se evalúa |
|---|---|
| **Utilidad pública** | Resuelve una pregunta real de gestión. |
| **Groundedness / citas** | La respuesta se apoya en un documento recuperado y citable. |
| **Costo / privacidad** | Usa capa gratuita; no expone datos sensibles. |
| **Reproducibilidad** | Corre sin red (modo caché) de principio a fin. |
| **Uso responsable** | Incluye la ficha y declara la supervisión humana. |

> **Entregable:** notebook reproducible (modo caché) + ficha de uso responsable + demo del RAG.